# Лабораторная 2 Численные методы

Вариант 5

Рассмотрим набор различных точек на отрезке $[a,b]:\ x_0 < x_1 < \dots < x_n$, в которых заданы значения функции $f(x)$ так, что $f_i=f(x_i), i=0,1,\dots ,n$. Требуется восстановить значение $f(x)$ и в других точках $x \in [a,b]$.


Для построения таблицы $(x_i, f_i)$ дано:

$[a,b] = [0.35; 1.35]$,  $x_i=0.35 + i \cdot h,\ h=\frac{1}{10},\ i=0,1,\dots,n$, где $n=10$.

$f(x)=0.35 e^x + 0.65 \cos x$


Необходимо:
1. Построить элемент наилучшего приближения методом наименьших квадратов.
2. Построить интерполяционный многочлен Лагранжа.
3. Построить остаток интерполирования в форме Ньютона.
4. Минимизировать остаток интерполирования через точки многочлена Чебышева.

Проверить результат, сравнив истинные значения функции и значения получившегося приближения функции в точках восстановления: $x^*=0.41(6),\ x^{**}=0.9,\ x^{***}=1,01(6)$

### Вычислим известные значения функций в узлах $x_i$:

In [11]:
import math

def foo(x: float)->float:
    return 0.35 * math.exp(x) + 0.65 * math.cos(x)

restore_points: list[float] = [5 / 12, 0.9, 79 / 600]

nodes: list[float] = [x / 100 for x in range(35, 136, 10)]
function_nodes: list[float] = [foo(x) for x in nodes]
print(restore_points)
function_nodes

[0.4166666666666667, 0.9, 0.13166666666666665]


[1.1072659053584362,
 1.1341998814507992,
 1.160779495592267,
 1.1878937592117498,
 1.2165477705824197,
 1.2478654429993352,
 1.2830923889120704,
 1.32359907245173,
 1.3708843549661207,
 1.426579570668569,
 1.4924532823544179]

|i |0|1|2|3|4|5|
|--|--|--|--|--|--|--|
|x_i |0.35|0.45|0.55|0.65|0.75|0.85|
|f(x_i) |1.1072659053584362| 1.1341998814507992 |  1.160779495592267 |1.1878937592117498|1.2165477705824197|1.2478654429993352|




|i |6|7|8|9|10|
|--|--|--|--|--|--|
|x_i |0.95|0.105|0.115|0.125|0.135|
|f(x_i) |1.2830923889120704| 1.32359907245173 |  1.3708843549661207 |1.426579570668569| 1.4924532823544179|

## 1. Построение элемента наилучшего приближения методом наименьших квадратов.

Алгоритм:

Строим полином степени $n=5$: $$P(x)= \sum^n_{i=0} c_i x^i$$

Коэффициенты $c_i$ находим из системы уравнений: $$\sum^n_{i=0} c_i (\sum^N_{j=0} p(x_j) x_j^{i+k}) = \sum^N_{j=0} p(x_j) f(x_j) x_j^k,\ k=\overline{0, n},\ p(x) \equiv 1$$
Поскольку $p(x) \equiv 1$, то система является системой линейной алгебраических уравнений. Решим её методом Гаусса.

Найдём погрешность вычислений в узлах по формуле $$\Delta f = (\sum_{j=0}^n (f(x_j) - P(x_j))^2)^{1/2}$$

In [12]:
def polynomial(coefficients:list[float], x: float)->float:
    y = 0.
    for i in range(len(coefficients)):
        y += coefficients[i] * x ** i
    return y

In [ ]:
def gaussian_method(a)->list[float]:
    def change_system_for_select_main_element(system, j):
        max_row = max(system[j:], key=lambda row: row[j])
        max_index = system.index(max_row)
        system[max_index], system[j] = system[j], system[max_index]
        m = 1 if max_index == 0 else -1
        return system, m
    def straight_stroke(a):
        n = len(a)
        for k in range(0,n):
            a, _ = change_system_for_select_main_element(a,k)
            for j in range(n,k-1,-1):
                a[k][j] /= a[k][k]
            for i in range(k+1,n):
                for j in range(n,k-1,-1):
                    a[i][j] = a[i][j] - a[i][k]*a[k][j]
        return a

    def reverse_stroke(a: list[list[float]])->list[float]:
        n = len(a)
        x = [0.] * n
        x[n-1] = a[n-1][n]
        for i in range(n-2,-1,-1):
            s = 0
            for j in range(i+1,n):
                s+=a[i][j]*x[j]
            x[i] = a[i][n] - s
        return x
    
    return reverse_stroke(straight_stroke(a))

In [27]:
import numpy as np

def least_squared_method(nodes: list[float], function_nodes: list[float], degree: int)->list[float]:
    m = degree + 1
    matrix = [[0.] * (m+1) for _ in range(m)]
    for i in range(m):
        for j in range(m):
            matrix[i][j] += sum(x ** (i+j) for x in nodes)
        matrix[i][m] = sum(f * x**i for f, x in zip(function_nodes, nodes))
    
    coefficients = gaussian_method(matrix)
    return coefficients

least_squared_coefficients = least_squared_method(nodes, function_nodes, 5)
print("Coefficients of least squared polynomial:")
print(least_squared_coefficients)

print("Theoretical error:")
least_squared_theoretical_error = 0.
for x, f in zip(nodes, function_nodes):
    least_squared_theoretical_error += (f - polynomial(least_squared_coefficients, x)) ** 2
least_squared_theoretical_error = least_squared_theoretical_error ** (1/2)
print(least_squared_theoretical_error)

print("Practical error in recovery points:")
for i in restore_points:
    print(polynomial(least_squared_coefficients, i) - foo(i))


Coefficients of least squared polynomial:
[0.9999238900918774, 0.3506195922763826, -0.15191198427973746, 0.06107719478494508, 0.04001999328453564, 0.0028663875131070184]
Theoretical error:
6.459455648685436e-07
Practical error in recovery points:
3.3968086765590044e-07
2.1334799371608426e-07
-2.190828622206098e-05


Таким образом, многочлен, полученный методом наименьших квадратов, выглядит следующим образом: $$P(x) = 0.9999238900918774 + 0.3506195922763826  x - 0.15191198427973746  x^2+  0.06107719478494508 x^3 + 0.04001999328453564 x^4 + 0.0028663875131070184 x^5$$

Погрешность вычислений: $\Delta f = 6.459455648685436e-07$. Невязка многочлена в заданных точках $x^*, x^{**}, x^{***}:$

|Формула|Значение|
|------|----|
|$P(x^*)-f(x^*)$| $3.3968086765590044e-07$|
|$P(x^{**})-f(x^{**})$ | $2.1334799371608426e-07$ |
|$P(x^{***})-f(x^{***})$ | $-2.190828622206098e-05$ |

С помощью метода наименьших квадратов найдено приближённое значение функции в заданных точках. Наибольшая невязка равна $-2.190828622206098e-05$ в точке $x^{***}$.

## 2. Построение интерполяционного многочлена Лагранжа

Интерполяционный многочлен Лагранжа имеет вид:$$P_n(x) = \sum_{i=0}^n l_i(x) f(x_i)$$ $$l_i=\frac{(x-x_0) \dots (x-x_{i-1})(x-x_{i+1}) \dots (x-x_n))}{(x_i-x_0) \dots (x_i-x_{i-1})(x_i-x_{i+1}) \dots (x_i-x_n)}$$

In [35]:
def lagrange_polynomial(nodes: list[float], function_nodes: list[float], x: float)->float:
    P = 0.
    for i, f in zip(nodes, function_nodes):
        term = f
        for j in nodes:
            if i == j: continue
            term = term * (x - j) / (i - j)
        P += term
    return P

print("Theoretical error:")
lagrange_polynomial_theoretical_error = 0.
for x, f in zip(nodes, function_nodes):
    lagrange_polynomial_theoretical_error += (f - lagrange_polynomial(nodes, function_nodes, x)) ** 2
lagrange_polynomial_theoretical_error = lagrange_polynomial_theoretical_error ** (1/2)
print(lagrange_polynomial_theoretical_error)

print("Practical error in recovery points:")
for i in restore_points:
    print(lagrange_polynomial(nodes, function_nodes, i) - foo(i))

Theoretical error:
4.440892098500626e-16
Practical error in recovery points:
-6.217248937900877e-14
1.9984014443252818e-15
2.1254176196805474e-10


Погрешность вычислений: $\Delta f = 4.440892098500626e-16$. Невязка многочлена в заданных точках $x^*, x^{**}, x^{***}:$

|Формула|Значение|
|------|----|
|$P(x^*)-f(x^*)$| $-6.217248937900877e-14$|
|$P(x^{**})-f(x^{**})$ | $1.9984014443252818e-15$ |
|$P(x^{***})-f(x^{***})$ | $2.1254176196805474e-10$ |

С помощью метода наименьших квадратов найдено приближённое значение функции в заданных точках. Наибольшая невязка равна $2.1254176196805474e-10$ в точке $x^{***}$.

## 3. Построение остатка интерполирования методом Ньютона